# Reports as artifacts

Reports can be stored with a dataset, shared in an issue, passed to an
agent, or compared across runs. This chapter covers JSON, Markdown, the
Python dict, and the conditions under which serialization is deterministic. The field-level
reference is [](../appendix/report_contract.ipynb).

The fixture is small and planted, so every claim below can be checked:

In [ ]:
import json
import random

import polars as pl

import onceover

rng = random.Random(4)
n = 800

region_id = [rng.randrange(3) for _ in range(n)]
region_name = [["red", "green", "blue"][r] for r in region_id]
quantity = [rng.randrange(1, 11) for _ in range(n)]
double_quantity = [2 * q for q in quantity]

df = pl.DataFrame({
    "order_id": list(range(n)),
    "region_id": region_id,
    "region_name": region_name,
    "quantity": quantity,
    "double_quantity": double_quantity,
})

report = onceover.profile(df)
print(report.overview["headline"])

## JSON

`to_json()` returns the complete report — overview, columns, findings,
groups, selection summary, coverage — as one deterministic, schema-versioned
string. `report.data` parses it into a dict; `save(path)` writes it;
`to_json(pretty=True)` formats it for humans. The `schema_version` field
(currently `"0.2"`) identifies the report format. Evolution is usually additive — new fields appear,
existing fields keep their meaning — but the 0.1 → 0.2 revision of the
finding taxonomy was breaking by design, so code that reads reports should
check the version and tolerate unknown categories defensively.

## Determinism

The same data and configuration produce byte-identical JSON. This is a
property of the whole pipeline — ordered maps, seeded splits, stable sorts,
content-hash finding ids — and allows repeated runs to be compared. Different input tables can still
produce the same report, because the report summarizes the data. An opt-in
deadline can change which passes complete; coverage records the skipped
passes. Package versions can also change the report.

In [ ]:
first = onceover.profile(df)
second = onceover.profile(df, budget="standard")

print("byte-identical:", first.to_json() == second.to_json())
print("schema_version:", report.data["schema_version"])
print("selection:", report.data["selection"])

The `selection` block travels with the report: how many candidates the
detectors produced, how many distinct statements they consolidated into,
and how many were selected under the policy — the audit trail for the
ledger's size.

## Markdown

`to_markdown()` renders the blocks as a compact document — headline, ranked
findings with their basis and form, the related-finding groups, a column
table, and the coverage table. It is the format meant for pasting into a
pull request or handing to a person. In a notebook, the report object
renders itself this way automatically:

In [ ]:
print(report.to_markdown()[:800])

## Feeding an agent

An agent reading a report should quote findings the way the ledger wrote
them. The pattern below packs the essentials: the claim verbatim, the id
and category for routing, and a compact evidence summary — basis,
validation, and the scope numbers an agent needs before repeating a
finding:

In [ ]:
def brief(f):
    primary = max(f["evidence"], key=lambda e: e["scope"]["rows_evaluated"] or 0)
    return {
        "id": f["id"],
        "category": f["category"],
        "type": f.get("type"),
        "claim": f["claim"],
        "basis": primary["basis"],
        "validation": primary["validation"]["method"],
        "rows_evaluated": primary["scope"]["rows_evaluated"],
        "interpretation": f.get("interpretation"),
    }

digest = {
    "headline": report.overview["headline"],
    "top_findings": [brief(f) for f in report.findings[:4]],
    "coverage": report.coverage,
}
print(json.dumps(digest["top_findings"][0], indent=2))

Carry basis, scope, and validation with the claim so that a sampled or
model-based result is not presented as a full-data measurement. When no
finding is reported, check coverage: "the profiler did not run that detector" and "the detector ran
it and found nothing" are different statements, and the ledger cannot
distinguish them by itself.

## The Rust API

The same profiler is a Rust crate over Polars 0.55, for pipelines that
never leave Rust:

```toml
[dependencies]
onceover = "0.1"
polars = "0.55"
```

```rust
use onceover::{profile, Budget, ProfileConfig};
use polars::prelude::*;

fn main() -> PolarsResult<()> {
    let df = df!("id" => [1i64, 2, 3], "value" => [4.0, 5.0, 6.0])?;

    let cfg = ProfileConfig {
        budget: Budget::Deep,
        detectors: None,        // None: default detectors
        selection: Default::default(),
        deadline: None,
        sample_rows: None,
        max_predictors: None,
        max_pairs: None,
    };
    let report = profile(&df, &cfg);

    println!("{}", report.overview.headline);
    println!("{}", report.to_json());   // same schema, same determinism
    Ok(())
}
```

`ProfileConfig::default()` uses the `standard` budget and default detectors, the
default selection policy, and no deadline; the `selection` field is the
same `SelectionPolicy` that caps findings and rules in Python via
`max_findings`/`max_rules`. The report schema is identical across
languages, so a Rust service and a Python notebook produce comparable
artifacts from the same table.